In [3]:
from __future__ import annotations
from dataclasses import dataclass
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

from dotenv import load_dotenv, find_dotenv
import requests
import os

from pydantic import BaseModel, Field
from typing import Literal
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate

from tqdm import tqdm

from langchain_openai import ChatOpenAI
import json

from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, accuracy_score

In [38]:
@dataclass
class KNNState:
    df: pd.DataFrame
    feature_cols: List[str]
    medians: pd.Series
    scaler: StandardScaler
    feats_imputed: np.ndarray   # NEW: features after numeric coercion + median impute
    X_scaled: np.ndarray        # normalized features used for KNN
    nn: NearestNeighbors
    excel_offset: int

In [39]:

# 1) build_state: infer features = all columns except the label 'Outcome' (unless use_cols is provided)
def build_state(
    df: pd.DataFrame,
    use_cols: Optional[List[str]] = None,
    excel_header_on_row1: bool = True,
    label_col: str = "Outcome",
    zero_as_missing: bool = True,
) -> KNNState:
    """
    For Pima Indians Diabetes:
      - label_col defaults to 'Outcome'
      - features = all columns except label (unless use_cols is provided)
      - optional: treat zeros in certain medical fields as missing before median impute
    """
    if label_col not in df.columns:
        raise ValueError(f"Label column '{label_col}' not found in DataFrame.")

    feature_cols = use_cols if use_cols is not None else [c for c in df.columns if c != label_col]

    feats = df[feature_cols].apply(pd.to_numeric, errors="coerce")

    # Optionally treat known-zero-as-missing fields as NaN before impute (common in Pima)
    if zero_as_missing:
        # Common Pima fields that use 0 as "missing"
        zero_missing_cols = [c for c in feature_cols if c.lower() in
                             {"glucose", "bloodpressure", "skinthickness", "insulin", "bmi"}]
        for c in zero_missing_cols:
            feats.loc[feats[c] == 0, c] = np.nan

    medians = feats.median(numeric_only=True)
    feats = feats.fillna(medians)
    feats_imputed = feats.values

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(feats_imputed)

    nn = NearestNeighbors(metric="euclidean")
    nn.fit(X_scaled)

    excel_offset = 2 if excel_header_on_row1 else 1
    return KNNState(
        df=df,
        feature_cols=feature_cols,
        medians=medians,
        scaler=scaler,
        feats_imputed=feats_imputed,
        X_scaled=X_scaled,
        nn=nn,
        excel_offset=excel_offset,
    )

In [40]:
#find the closest neighbors

def closest_indices_for_row(
    row_idx: int,
    state: KNNState,
    k: int = 5,
    return_distances: bool = False,
) -> np.ndarray | Tuple[np.ndarray, np.ndarray]:
    """
    Given a row index from the same DataFrame used in `build_state`,
    return indices of the k closest rows (excluding the row itself).
    Optionally return distances as well.

    Returns:
      - indices: shape (k,) 0-based row indices
      - distances (optional): shape (k,) Euclidean distances in normalized space
    """
    # Query for k+1 (self + k neighbors), then drop self
    distances, indices = state.nn.kneighbors(
        state.X_scaled[row_idx].reshape(1, -1),
        n_neighbors=k + 1
    )
    inds = indices[0].tolist()
    dists = distances[0].tolist()

    # Remove self if present
    if row_idx in inds:
        j = inds.index(row_idx)
        inds.pop(j)
        dists.pop(j)

    # Trim in case self wasn't present for some reason
    inds = np.array(inds[:k], dtype=int)
    dists = np.array(dists[:k], dtype=float)

    return (inds, dists) if return_distances else inds

In [41]:
def pretty_print_row(row_idx: int, state: KNNState):
    """Prints the full Excel row (all columns) for the given row index."""
    row_df = pd.DataFrame([state.df.iloc[row_idx]], columns=state.df.columns)
    print(row_df.to_string(index=False))

In [42]:
#2) format_fewshot: emit only features + label for Pima (no reasoning columns exist)

def format_fewshot(row_idx: int, state: KNNState, label_col: str = "Outcome") -> dict:
    """
    Return a row as a few-shot example object suitable for Pima:
      - features: dict of all feature columns (state.feature_cols)
      - label:    value from 'Outcome'
    """
    if label_col not in state.df.columns:
        raise ValueError(f"Label column '{label_col}' not found in DataFrame.")

    row = state.df.iloc[row_idx]
    return {
        "features": row[state.feature_cols].to_dict(),
        "label": row[label_col],
    }

In [43]:
def render_few_shots_block(df: pd.DataFrame, neighbor_indices: list[int], state: KNNState,
                           label_col: str = "Outcome") -> str:
    """
    Build a text block for few-shot examples:
      features_json: {...}
      label: <0 or 1>
    """
    lines = []
    for idx in neighbor_indices:
        ex = format_fewshot(idx, state, label_col=label_col)
        features_json = json.dumps(ex["features"], ensure_ascii=False)
        lines.append(
            f"features_json: {features_json}\n"
            f"label: {ex['label']}\n"
        )
    return "\n".join(lines).strip()

In [60]:
#all configurations

#EXCEL_PATH = "../data/Patients_toy.xlsx"       # adjust path if neede
EXCEL_PATH = "../data/diabetes.csv"
SHEET = 0                                           # or a sheet name = "openai/gpt-oss-20b"
MODEL = "gpt-4o-mini"   # any OpenAI chat model
TEMPERATURE = 0.0                                   # deterministic classification
TARGET_COL = "Outcome"                               # ground-truth label in your file
OUT_CSV = "../outputs/Llama8B-Diabetes-RAIL.csv"

In [53]:
# OpenAI API access: put OPENAI_API_KEY in your environment or in a local .env file
# (see .env.example). Never commit your key.
load_dotenv(find_dotenv())
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is not set")

In [54]:
# strcutured output and getting the prompt

class IecHsPrediction(BaseModel):
    rationale: str = Field(
        ..., description="key supporting evidence for outcome to be positive as well as contradicting factors. Followed by overall reasoning for what should be the prediction for dibetest Outcome."
    )
    outcome: Literal[0, 1] = Field(..., description="Predicted outcome class label (0 or 1).")
    confidence: float = Field(
        ..., ge=0.0, le=1.0,
        description="Model self-reported confidence between 0 and 1."
    )
   

#parser = JsonOutputParser(pydantic_object=IecHsPrediction)
parser = PydanticOutputParser(pydantic_object=IecHsPrediction)

system_msg = """You are a careful clinical classifier. 
Given a single patient's anonymized features, predict whether diabetes is 0 or 1.
Only use the provided features—do not invent data.

Output MUST be valid JSON matching this schema:
- rationale: short reasoning for prediction (3-5 sentences)
- outcome: 0 or 1
- confidence: a number in [0, 1]
"""

human_template = """
### Few-shot examples
{few_shots_block}

### Patient to classify
Patient features (key → value) as JSON:
{patient_json}

Instructions:
1) Consider all the supporting as well as contradicting evidences and reason about what the prediction of diabetes should be.
2) Predict outcome (0 or 1).
3) Return ONLY JSON, no extra text.

Use this format:
{format_instructions}

Output MUST be valid JSON matching this schema:
- rationale: short reasoning for prediction (3-5 sentences)
- outcome: 0 or 1
- confidence: a number in [0, 1]
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_msg), ("human", human_template)]
)

# IMPORTANT: inject the raw JSON schema as a *variable*, not as literal text
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

In [55]:
llm = ChatOpenAI(
        model=MODEL,
        temperature=0,
        max_retries=2,
        max_tokens=2048,
        api_key=api_key,
    )

In [56]:
chain = prompt | llm | parser

In [57]:
df = pd.read_csv(EXCEL_PATH)
#selected_cols = ["CRP", "Diagnosis", "Ferritin", "Tocilizumab given"]
#state = build_state(df, use_cols=selected_cols)
#state = build_state(df)  # uses first 24 columns (A..X) by default
state = build_state(df, label_col="Outcome", zero_as_missing=True)
test_row = 11
nn_inds, nn_dists = closest_indices_for_row(test_row, state, k=5, return_distances=True)
print(render_few_shots_block(df, nn_inds, state))

features_json: {"Pregnancies": 10.0, "Glucose": 179.0, "BloodPressure": 70.0, "SkinThickness": 0.0, "Insulin": 0.0, "BMI": 35.1, "DiabetesPedigreeFunction": 0.2, "Age": 37.0}
label: 0.0

features_json: {"Pregnancies": 9.0, "Glucose": 170.0, "BloodPressure": 74.0, "SkinThickness": 31.0, "Insulin": 0.0, "BMI": 44.0, "DiabetesPedigreeFunction": 0.403, "Age": 43.0}
label: 1.0

features_json: {"Pregnancies": 11.0, "Glucose": 138.0, "BloodPressure": 76.0, "SkinThickness": 0.0, "Insulin": 0.0, "BMI": 33.2, "DiabetesPedigreeFunction": 0.42, "Age": 35.0}
label: 0.0

features_json: {"Pregnancies": 8.0, "Glucose": 151.0, "BloodPressure": 78.0, "SkinThickness": 32.0, "Insulin": 210.0, "BMI": 42.9, "DiabetesPedigreeFunction": 0.516, "Age": 36.0}
label: 1.0

features_json: {"Pregnancies": 7.0, "Glucose": 178.0, "BloodPressure": 84.0, "SkinThickness": 0.0, "Insulin": 0.0, "BMI": 39.9, "DiabetesPedigreeFunction": 0.331, "Age": 41.0}
label: 1.0


In [71]:
results = []
errored_rows = []  # to keep track of rows where exceptions happened

#selected_cols = ["CRP", "Diagnosis", "Ferritin", "Tocilizumab given"]

#state = build_state(df, use_cols=selected_cols)

#state = build_state(df)  # uses first 24 columns (A..X) by default
state = build_state(df, label_col="Outcome", zero_as_missing=True)

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Scoring patients"):
    try: 
        nn_inds, nn_dists = closest_indices_for_row(idx, state, k=0, return_distances=True)
        few_shot_block = render_few_shots_block(df, nn_inds, state)
        record = format_fewshot(idx, state)
        patient_features = record["features"]
        # Safety: ensure we never leak the ground-truth label to the model
        assert TARGET_COL not in patient_features
        #print(f"patient features are {patient_features}")
        #print(f"few shot are {few_shot_block}")

        pred: IecHsPrediction = chain.invoke({
            "few_shots_block": few_shot_block,
            "patient_json": json.dumps(patient_features, ensure_ascii=False)
        })
        #print(f"prediction is {pred['iec_hs']}")
        #print(f"prediction is {pred.outcome} with confidence {pred.confidence}")
        print(f"prediction for row {idx} is {pred.outcome} with confidence {pred.confidence} and true label is {row.get(TARGET_COL, None)}")
        results.append({
                "row_index": idx,
                "iec_hs_pred": int(pred.outcome),
                "confidence": float(pred.confidence),
                "rationale": pred.rationale,
                "iec_hs_true": row.get(TARGET_COL, None)
            })
    except Exception as e:
        errored_rows.append((idx, str(e)))  # store row index + error message
        print(e)
        print(f"prediction is SKIPPED")
        continue  # skip to next row

Scoring patients:   0%|          | 1/768 [00:02<36:23,  2.85s/it]

prediction for row 0 is 1 with confidence 0.7 and true label is 1.0


Scoring patients:   0%|          | 2/768 [00:04<30:37,  2.40s/it]

prediction for row 1 is 0 with confidence 0.8 and true label is 0.0


Scoring patients:   0%|          | 3/768 [00:06<25:47,  2.02s/it]

...

prediction for row 765 is 1 with confidence 0.7 and true label is 0.0


Scoring patients: 100%|█████████▉| 767/768 [24:27<00:01,  1.98s/it]

prediction for row 766 is 1 with confidence 0.7 and true label is 1.0


Scoring patients: 100%|██████████| 768/768 [24:29<00:00,  1.91s/it]

prediction for row 767 is 0 with confidence 0.6 and true label is 0.0


In [72]:
out_df = pd.DataFrame(results)

# Keep only rows where both are non-null
valid = out_df["iec_hs_pred"].notna() & out_df["iec_hs_true"].notna()
y_true = out_df.loc[valid, "iec_hs_true"].astype(int)
y_pred = out_df.loc[valid, "iec_hs_pred"].astype(int)

# Overall metrics (positive class = 1)
accuracy   = accuracy_score(y_true, y_pred)
precision1 = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
recall1    = recall_score(y_true, y_pred,    pos_label=1, zero_division=0)
f1_pos     = f1_score(y_true, y_pred,        pos_label=1, zero_division=0)

# Negative-class metrics (label = 0)
precision0 = precision_score(y_true, y_pred, pos_label=0, zero_division=0)
recall0    = recall_score(y_true, y_pred,    pos_label=0, zero_division=0)
f1_neg     = f1_score(y_true, y_pred,        pos_label=0, zero_division=0)

# Macro F1 (unweighted mean of per-class F1s)
f1_macro   = f1_score(y_true, y_pred, average="macro", zero_division=0)

print(f"Accuracy:     {accuracy:.3f}")
print(f"Pos P/R/F1:   P={precision1:.3f}  R={recall1:.3f}  F1={f1_pos:.3f}")
print(f"Neg P/R/F1:   P={precision0:.3f}  R={recall0:.3f}  F1={f1_neg:.3f}")
print(f"Macro F1:     {f1_macro:.3f}")

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print("\nConfusion Matrix (rows = true, cols = pred):")
print("         Pred 0   Pred 1")
print(f"True 0   {cm[0,0]:6}   {cm[0,1]:6}")
print(f"True 1   {cm[1,0]:6}   {cm[1,1]:6}")

Accuracy:     0.660
Pos P/R/F1:   P=0.509  R=0.776  F1=0.614
Neg P/R/F1:   P=0.833  R=0.598  F1=0.696
Macro F1:     0.655

Confusion Matrix (rows = true, cols = pred):
         Pred 0   Pred 1
True 0      299      201
True 1       60      208


In [64]:
# Save results
out_df = pd.DataFrame(results)
print(out_df.columns.tolist())

#print(out_df)
out_df.to_csv(OUT_CSV, index=False)

['row_index', 'iec_hs_pred', 'confidence', 'rationale', 'iec_hs_true']
